In [64]:
import pandas as pd
import requests
from datetime import datetime
from statsmodels.tsa.seasonal import MSTL

#### I. MTA Daily Ridership and Traffic

In [65]:
import requests

# Use the correct Socrata API endpoint (resource endpoint, not views)
url = "https://data.ny.gov/resource/sayj-mze2.json"

params = {
    "$where": "date >= '2023-01-01'" +
              " AND mode IN ('Bus', 'LIRR', 'MNR', 'Subway')",
    "$order": "date ASC",
    "$limit": 100000
}
response = requests.get(url, params=params)
data = response.json()

In [66]:
# Convert to DataFrame
df = pd.DataFrame(data)
df['date'] = pd.to_datetime(df['date'])
df['count'] = pd.to_numeric(df['count'], errors='coerce')

In [67]:
df.sample(10)

,date,mode,count
1815,2024-03-29,Subway,3372535
3510,2025-05-27,MNR,243011
2189,2024-07-01,LIRR,216906
3141,2025-02-24,LIRR,234816
3997,2025-09-26,LIRR,306735
3227,2025-03-17,Subway,3751313
1744,2024-03-12,Bus,1370319
4056,2025-10-11,Bus,859727
1478,2024-01-05,MNR,173362
3799,2025-08-07,Subway,3944318


In [68]:
df['mode'].unique()

array(['Bus', 'LIRR', 'MNR', 'Subway'], dtype=object)

In [69]:
abb_to_name = {
    "LIRR": "Long Island Rail Road",
    "Subway": "Subway",
    "Bus": "Bus",
    "MNR": "Metro-North Railroad"
}

In [70]:
df['mode'] = df['mode'].map(abb_to_name)

df['mode'].unique()

array(['Bus', 'Long Island Rail Road', 'Metro-North Railroad', 'Subway'],
      dtype=object)

In [71]:
pd.options.display.float_format = '{:.2f}'.format

def seasonally_adjust_group(group_df):
    """Apply MSTL to a single mode's time series (weekly + yearly seasonality)."""
    group_df = group_df.sort_values('date').set_index('date')
    
    # Ensure continuous daily index
    group_df = group_df.asfreq('D')
    
    # Interpolate any gaps
    group_df['count'] = group_df['count'].interpolate(method='linear')
    
    # MSTL: 7-day (weekly) and 365-day (yearly) seasonality
    mstl = MSTL(group_df['count'], periods=[7, 365])
    result = mstl.fit()
    
    # Remove both seasonal components
    group_df['count_sa'] = group_df['count'] - result.seasonal.sum(axis=1)
    
    return group_df.reset_index()

# Apply seasonal adjustment to each mode
df = (
    df.groupby('mode', group_keys=False)
    .apply(seasonally_adjust_group)
)

In [72]:
df

,date,mode,count,count_sa
0,2023-01-01,Bus,475226,1344741.19
1,2023-01-02,Bus,739507,1122422.93
2,2023-01-03,Bus,1268913,1179553.40
3,2023-01-04,Bus,1391449,1219909.10
4,2023-01-05,Bus,1378698,1225090.48
...,...,...,...,...
1106,2026-01-11,Subway,1978524,3567181.68
1107,2026-01-12,Subway,3769984,3633566.35
1108,2026-01-13,Subway,4095328,3550306.77
1109,2026-01-14,Subway,4104686,3662210.95


In [73]:
df = df.sort_values(['mode', 'date']).set_index('date')
df['count_ma90'] = df.groupby('mode')['count'].transform(lambda x: x.rolling('90D', min_periods=1).mean())
df['count_sa_ma90'] = df.groupby('mode')['count_sa'].transform(lambda x: x.rolling('90D', min_periods=1).mean())
df = df.reset_index()

In [74]:
# order the df by mode from highest to lowest average daily ridership
mode_order = df.groupby('mode')['count'].mean().sort_values(ascending=False).index
df['mode'] = pd.Categorical(df['mode'], categories=mode_order, ordered=True)
df = df.sort_values(['mode', 'date'])

In [76]:
df

,date,mode,count,count_sa,count_ma90,count_sa_ma90
3333,2023-01-01,Subway,1675507,3407188.96,1675507.00,3407188.96
3334,2023-01-02,Subway,1938154,2881522.55,1806830.50,3144355.76
3335,2023-01-03,Subway,3175777,3012433.67,2263146.00,3100381.73
3336,2023-01-04,Subway,3413052,3108331.57,2550622.50,3102369.19
3337,2023-01-05,Subway,3428520,3108516.76,2726202.00,3103598.70
...,...,...,...,...,...,...
3328,2026-01-11,Metro-North Railroad,84855,194699.55,193689.94,193907.17
3329,2026-01-12,Metro-North Railroad,200032,196338.89,193132.20,193966.06
3330,2026-01-13,Metro-North Railroad,220239,192269.66,192794.48,193993.71
3331,2026-01-14,Metro-North Railroad,223140,201637.71,192505.13,194089.43


In [75]:
df.to_csv("data/processed/daily_ridership.csv", index=False)